# Step 3 Demo — HelixFold Structure Prediction (+ optional Relax)

This notebook shows the minimal workflow for:
1. checking HelixFold weights
2. predicting PDBs for a tiny sequence CSV
3. (optional) PyRosetta FastRelax

> HelixFold needs the `helix` / `helix_test` conda env.  
> Relax needs the `rosetta` conda env.  
> Switch kernels / terminals accordingly before running GPU inference.


In [ ]:
from pathlib import Path
import pandas as pd

STEP_DIR = Path('.').resolve()
print('Working directory:', STEP_DIR)
assert (STEP_DIR / 'infer_batch.py').exists(), 'Please open/run this notebook from 03_structure_prediction/'

WEIGHTS = STEP_DIR / 'weights' / 'helixfold-single.pdparams'
print('Weights exist:', WEIGHTS.exists(), '->', WEIGHTS)
print('Resolved:', WEIGHTS.resolve() if WEIGHTS.exists() else 'MISSING')


In [ ]:
# Tiny example sequences (edit freely)
EXAMPLE_CSV = STEP_DIR / 'examples' / 'sample_sequences.csv'
df = pd.read_csv(EXAMPLE_CSV)
display(df)

# For a very fast notebook demo, keep only the first sequence
DEMO_CSV = STEP_DIR / 'outputs' / 'notebook_demo_input.csv'
DEMO_CSV.parent.mkdir(parents=True, exist_ok=True)
df.head(1).to_csv(DEMO_CSV, index=False)
print('Wrote demo CSV:', DEMO_CSV)


## A) HelixFold batch inference

Run in a terminal with the HelixFold environment activated:

```bash
conda activate helix
export LD_LIBRARY_PATH="$CONDA_PREFIX/lib:$LD_LIBRARY_PATH"
cd 03_structure_prediction
CUDA_VISIBLE_DEVICES=0 python infer_batch.py   --csv_file outputs/notebook_demo_input.csv   --output_dir outputs/pdb/notebook_demo   --init_model weights/helixfold-single.pdparams
```

Or execute the next cell **only if this notebook kernel already has paddle + HelixFold deps**.


In [ ]:
# Optional in-kernel inference (requires HelixFold / paddle environment)
RUN_IN_KERNEL = False  # set True only in helix env

if RUN_IN_KERNEL:
    import os
    os.chdir(STEP_DIR)
    from infer_batch import run_batch
    from types import SimpleNamespace
    args = SimpleNamespace(
        csv_file=str(DEMO_CSV),
        output_dir=str(STEP_DIR / 'outputs' / 'pdb' / 'notebook_demo'),
        init_model=str(WEIGHTS),
        skip_exist=True,
    )
    run_batch(args)
else:
    print('Skipped in-kernel HelixFold run. Use the terminal command above.')


In [ ]:
# Inspect predicted PDBs if present
pdb_dir = STEP_DIR / 'outputs' / 'pdb' / 'notebook_demo'
pdbs = sorted(pdb_dir.glob('*.pdb')) if pdb_dir.exists() else []
print(f'Found {len(pdbs)} PDB(s) in {pdb_dir}')
for p in pdbs[:5]:
    print('-', p.name, f'({p.stat().st_size} bytes)')
    print('  head:')
    print('\n'.join(p.read_text().splitlines()[:8]))


## B) Optional PyRosetta FastRelax

```bash
conda activate rosetta
cd 03_structure_prediction
python relax.py   --input_dir outputs/pdb/notebook_demo   --output_dir outputs/relaxed/notebook_demo
```


In [ ]:
# Optional in-kernel relax (requires pyrosetta)
RUN_RELAX_IN_KERNEL = False

if RUN_RELAX_IN_KERNEL:
    from relax import run_relax
    run_relax(
        input_dir=STEP_DIR / 'outputs' / 'pdb' / 'notebook_demo',
        output_dir=STEP_DIR / 'outputs' / 'relaxed' / 'notebook_demo',
        workers=2,
    )
else:
    print('Skipped in-kernel relax. Use the rosetta terminal command above.')


## Done

Relaxed PDBs can be passed to downstream AMP ranking models (POAP Step 4 / TIGER activity models).
